# Alpha 返回字段与曲线分析

> Notebook version: `2026-06-08-v3-report-layout`。本文件不依赖 `matplotlib` 或 `plotly`，图表使用内置 HTML/SVG 输出。

这个 notebook 用来完成一个小闭环：

1. 获取前 5 个符合条件的 Alpha。
2. 查看 `get_alphas_full()` 返回字段和表达式。
3. 调用单个 Alpha 详情接口 `/alphas/{alpha_id}`。
4. 调用 recordsets 接口，拉取 `pnl`、`daily-pnl`、`sharpe`、`turnover`、`yearly-stats`。
5. 画快照指标图、散点图、时间序列曲线和年度统计图。
6. 可选把结果保存到本地，方便后续复盘。

In [86]:
from pathlib import Path
import importlib
import inspect
import json
import sys
import time
from datetime import date

cwd = Path.cwd().resolve()
project_candidates = [
    cwd,
    cwd.parent,
    cwd / "consultant",
    cwd.parent / "consultant",
]
PROJECT_ROOT = next(
    (candidate for candidate in project_candidates if (candidate / "src" / "consultant_core").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(f"Cannot find consultant project root from working directory: {cwd}")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd
from IPython.display import HTML, display

from consultant_core.alpha_report import (
    normalize_date_filter,
    render_alpha_report,
)

import consultant_core.machine_lib as machine_lib
machine_lib = importlib.reload(machine_lib)

from consultant_core.machine_lib import (
    get_alphas_full,
    get_alpha_detail,
    get_alpha_recordsets,
    get_alpha_recordset,
    login,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

In [87]:
NOTEBOOK_VERSION = "2026-06-08-v3-report-layout"
print("notebook version:", NOTEBOOK_VERSION)
print("chart backend: built-in HTML/SVG, no matplotlib required")
print("project root:", PROJECT_ROOT)

notebook version: 2026-06-08-v3-report-layout
chart backend: built-in HTML/SVG, no matplotlib required
project root: D:\code\WorldQuant Brain\consultant


## 0. 参数区

优先只改这个 cell。默认只拉前 5 个符合条件的 Alpha，避免频繁请求平台。

In [88]:
REGION = "USA"
USAGE = "submit"
SHARPE_TH = 1.5
FITNESS_TH = 1
LIMIT = 5
ALPHA_ORDER = "-dateCreated"

# LIMIT=0 表示全选。ALPHA_ORDER=-dateCreated 表示按创建时间倒序，优先看最新 Alpha。

# 日期过滤：DATE_FILTER_ENABLED=False 表示不限制时间，全选。
# START_DATE 默认当前日期；启用过滤后会作为开始时间。END_DATE 留空表示没有结束时间上限。
DATE_FILTER_ENABLED = False
START_DATE = date.today().isoformat()
END_DATE = ""

# 如果指定 SELECTED_ALPHA_ID，就分析这个 Alpha；否则默认分析返回结果的第 SELECTED_ALPHA_INDEX 行。
SELECTED_ALPHA_ID = None
SELECTED_ALPHA_INDEX = 0

# 需要拉取的曲线/统计表。不同 Alpha 可能支持不同 recordset，缺失时会自动跳过。
DESIRED_RECORDSETS = ["pnl", "daily-pnl", "sharpe", "turnover", "yearly-stats"]

# 平台有限流，recordset 之间留一点间隔更稳。
REQUEST_PAUSE_SECONDS = 1.0

# 是否保存到本地。默认关闭，避免无意写入文件。
SAVE_LOCAL_FILES = True
SAVE_DIR = PROJECT_ROOT / "data_catalog" / "alpha_analysis"

## 1. 获取 Alpha 列表

这里复用同一个登录 session，减少触发平台限流的概率。

In [89]:
s = login()
print("session: authenticated")

start_date = normalize_date_filter(START_DATE) if DATE_FILTER_ENABLED else None
end_date = normalize_date_filter(END_DATE) if DATE_FILTER_ENABLED else None
alpha_limit = None if LIMIT == 0 else LIMIT
print("date filter:", start_date or "ALL", "->", end_date or "ALL")
print("limit:", alpha_limit or "ALL")
print("order:", ALPHA_ORDER)

alphas = get_alphas_full(
    start_date=start_date,
    end_date=end_date,
    sharpe_th=SHARPE_TH,
    fitness_th=FITNESS_TH,
    region=REGION,
    usage=USAGE,
    limit=alpha_limit,
    session=s,
    order=ALPHA_ORDER,
)

if alphas.empty:
    raise RuntimeError("No alpha rows returned. Adjust REGION / SHARPE_TH / FITNESS_TH / LIMIT.")

display_cols = [
    "alpha_id", "name", "dateCreated", "sharpe", "fitness", "turnover",
    "margin", "longCount", "shortCount", "instrumentCount", "decay",
    "next_decay", "region", "universe", "neutralization", "direction", "status",
]
alphas[display_cols]

session: authenticated
date filter: ALL -> ALL
limit: 5
order: -dateCreated


,alpha_id,name,dateCreated,sharpe,fitness,turnover,margin,longCount,shortCount,instrumentCount,decay,next_decay,region,universe,neutralization,direction,status
0,88LzKw7v,None,2026-06-07T22:13:23-04:00,2.02,1.39,0.0859,0.001371,979,990,1969,6,6,USA,TOP3000,SUBINDUSTRY,positive,UNSUBMITTED
1,58vwqGz6,None,2026-06-07T22:10:50-04:00,2.13,1.67,0.1718,0.001233,330,323,653,256,256,USA,TOP3000,SUBINDUSTRY,positive,UNSUBMITTED
2,pw761RQg,None,2026-06-07T22:09:33-04:00,1.59,1.07,0.1617,0.000898,335,338,673,128,128,USA,TOP3000,SUBINDUSTRY,positive,UNSUBMITTED
3,d5QxljOv,None,2026-06-07T22:09:10-04:00,1.91,1.38,0.1093,0.001196,888,952,1840,6,6,USA,TOP3000,SUBINDUSTRY,positive,UNSUBMITTED
4,RRrpk77o,None,2026-06-07T22:09:10-04:00,1.63,1.10,0.1610,0.000918,336,338,674,512,512,USA,TOP3000,SUBINDUSTRY,positive,UNSUBMITTED


## 2. 查看表达式

`expression` 是后续优化、剪枝、分类、提交前检查最重要的字段。

In [90]:
alphas[["alpha_id", "expression"]]

,alpha_id,expression
0,88LzKw7v,"trade_when(ts_corr(close, volume, 20) < 0, group_zscore(ts_rank(anl4_fs_detail_estimates_advanced_af_nd_ptp_low / enterprise_value, 120),densify(bucket(grou..."
1,58vwqGz6,"trade_when(ts_corr(close, volume, 20) < 0, ts_ir(anl4_cfo_low / fnd6_newqv1300_spcepq, 22), abs(returns) > 0.1)"
2,pw761RQg,"trade_when(ts_regression(returns, ts_step(20), 20, lag = 0, rettype = 2) > 0, ts_ir(anl4_cfo_low / fnd6_newqv1300_spcepq, 22), abs(returns) > 0.1)"
3,d5QxljOv,"trade_when(ts_arg_max(volume, 5) == 0, ts_zscore(anl4_afv4_eps_mean / enterprise_value, 252), abs(returns) > 0.1)"
4,RRrpk77o,"trade_when(ts_regression(returns, ts_step(20), 20, lag = 0, rettype = 2) > 0, ts_ir(anl4_cfo_low / fnd6_newqv1300_spcepq, 22), abs(returns) > 0.1)"


## 3. 选择要深入分析的 Alpha

默认选择列表第一个 Alpha。也可以在参数区填写 `SELECTED_ALPHA_ID`。

In [91]:
if SELECTED_ALPHA_ID:
    selected = alphas[alphas["alpha_id"] == SELECTED_ALPHA_ID]
    if selected.empty:
        raise RuntimeError(f"SELECTED_ALPHA_ID not found in current alpha list: {SELECTED_ALPHA_ID}")
    alpha_id = SELECTED_ALPHA_ID
    selected_row = selected.iloc[0]
else:
    selected_row = alphas.iloc[SELECTED_ALPHA_INDEX]
    alpha_id = selected_row["alpha_id"]

print("selected alpha_id:", alpha_id)

selected alpha_id: 88LzKw7v


## 4. 获取单个 Alpha 详情

`/alphas/{alpha_id}` 会返回更多平台原始信息，例如 `is`、`settings`、`regular`、`train/test/prod` 等。

In [92]:
detail = get_alpha_detail(alpha_id, session=s)

print("top-level keys:", sorted(detail.keys()))
print("is keys:", sorted((detail.get("is") or {}).keys()))
print("settings keys:", sorted((detail.get("settings") or {}).keys()))
print("regular keys:", sorted((detail.get("regular") or {}).keys()))

top-level keys: ['author', 'category', 'classifications', 'color', 'competitions', 'dateCreated', 'dateModified', 'dateSubmitted', 'favorite', 'grade', 'hidden', 'id', 'is', 'name', 'origin', 'os', 'osmosisPoints', 'prod', 'pyramidThemes', 'pyramids', 'regular', 'settings', 'stage', 'status', 'tags', 'team', 'test', 'themes', 'train', 'type']
is keys: ['bookSize', 'checks', 'drawdown', 'fitness', 'investabilityConstrained', 'longCount', 'margin', 'pnl', 'returns', 'riskNeutralized', 'sharpe', 'shortCount', 'startDate', 'turnover']
settings keys: ['decay', 'delay', 'endDate', 'instrumentType', 'language', 'maxPosition', 'maxTrade', 'nanHandling', 'neutralization', 'pasteurization', 'region', 'startDate', 'truncation', 'unitHandling', 'universe', 'visualization']
regular keys: ['code', 'description', 'operatorCount']


## 5. 查看可用 recordsets

当前封装接口：

| 函数 | 平台接口 | 用途 |
|---|---|---|
| `get_alphas_full(...)` | `/users/self/alphas` | 获取 Alpha 列表和常用指标 |
| `get_alpha_detail(alpha_id)` | `/alphas/{alpha_id}` | 获取单个 Alpha 详情 |
| `get_alpha_recordsets(alpha_id)` | `/alphas/{alpha_id}/recordsets` | 查看该 Alpha 有哪些曲线或统计表 |
| `get_alpha_recordset(alpha_id, name)` | `/alphas/{alpha_id}/recordsets/{name}` | 拉取某一个 recordset 并转成 DataFrame |

不同 Alpha 支持的 recordsets 不一定完全一致，所以后续会按可用列表自动跳过缺失项。

In [93]:
recordsets = get_alpha_recordsets(alpha_id, session=s)
recordsets_df = pd.DataFrame(recordsets.get("results", []))
recordsets_df

,name,title
0,pnl,PnL
1,sharpe,Sharpe
2,turnover,Turnover
3,daily-pnl,Daily PnL
4,yearly-stats,Yearly Stats


## 6. 拉取曲线和统计表

这里会先看平台返回的可用 recordsets，再拉取 `DESIRED_RECORDSETS` 中存在的项目。任何单个 recordset 失败都不会中断整个 notebook。

In [94]:
available_recordsets = set(recordsets_df["name"].dropna()) if "name" in recordsets_df.columns else set()
series_frames = {}

for name in DESIRED_RECORDSETS:
    if available_recordsets and name not in available_recordsets:
        print(name, "skipped: not listed by platform")
        continue
    try:
        df = get_alpha_recordset(alpha_id, name, session=s)
        series_frames[name] = df
        print(name, df.shape, df.columns.tolist())
    except Exception as exc:
        print(name, "not available:", exc)
    time.sleep(REQUEST_PAUSE_SECONDS)

print("loaded recordsets:", sorted(series_frames))

pnl (2494, 4) ['date', 'pnl', 'risk-neutralized-pnl', 'investability-constrained-pnl']
daily-pnl (2494, 2) ['date', 'pnl']
sharpe (2494, 2) ['date', 'sharpe']
turnover (2493, 2) ['date', 'turnover']
yearly-stats (10, 12) ['year', 'pnl', 'bookSize', 'longCount', 'shortCount', 'turnover', 'sharpe', 'returns', 'drawdown', 'margin', 'fitness', 'stage']
loaded recordsets: ['daily-pnl', 'pnl', 'sharpe', 'turnover', 'yearly-stats']


## 7. Alpha Report

This section renders the selected alpha in a platform-like layout: code, aggregate data, simulation settings, failed or warning checks, chart data, yearly aggregate table, and properties.


In [95]:
report_html = render_alpha_report(
    alpha_id=alpha_id,
    detail=detail,
    recordsets_df=recordsets_df,
    series_frames=series_frames,
    selected_row=selected_row,
)
display(HTML(report_html))


Instrument Type,EQUITY
Region,USA
Universe,TOP3000
Language,FASTEXPR
Decay,6
Delay,1
Truncation,0.08
Neutralization,SUBINDUSTRY
Pasteurization,ON
NaN Handling,ON
Unit Handling,VERIFY


## 8. 可选保存到本地

默认 `SAVE_LOCAL_FILES = False`。如果要保存，把参数区改成 `True` 后重新运行本 cell。

In [96]:
if SAVE_LOCAL_FILES:
    output_dir = SAVE_DIR / str(alpha_id)
    output_dir.mkdir(parents=True, exist_ok=True)
    parameters = {
        "notebook_version": NOTEBOOK_VERSION,
        "alpha_id": str(alpha_id),
        "platform_url": f"https://platform.worldquantbrain.com/alpha/{alpha_id}",
        "region": REGION,
        "usage": USAGE,
        "sharpe_th": SHARPE_TH,
        "fitness_th": FITNESS_TH,
        "limit": LIMIT,
        "effective_limit": alpha_limit,
        "order": ALPHA_ORDER,
        "date_filter_enabled": DATE_FILTER_ENABLED,
        "start_date": start_date,
        "end_date": end_date,
        "desired_recordsets": DESIRED_RECORDSETS,
    }
    (output_dir / "parameters.json").write_text(json.dumps(parameters, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_dir / "alpha_detail.json").write_text(json.dumps(detail, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    (output_dir / "alpha_report.html").write_text(report_html, encoding="utf-8")
    alphas.to_excel(output_dir / "alphas.xlsx", index=False)
    recordsets_df.to_excel(output_dir / "recordsets.xlsx", index=False)
    for name, df in series_frames.items():
        safe_name = name.replace("/", "_")
        df.to_csv(output_dir / f"{safe_name}.csv", index=False, encoding="utf-8-sig")
    print("saved to:", output_dir)
else:
    print("SAVE_LOCAL_FILES is False; no files written")

saved to: D:\code\WorldQuant Brain\consultant\data_catalog\alpha_analysis\88LzKw7v
